In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )



In [2]:
df = pd.read_csv("../data/application_train.csv")
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
def create_files_nulls_per_colmun(data_frame,table_name):
    nulls=data_frame.isna().sum()
    nulls.head()
    nulls.to_csv("dumps_from_notebooks/" + "null_count_" + table_name,index=True)
    porcentaje_of_nulls = (nulls * 100) / len(data_frame)
    porcentaje_of_nulls.head()
    porcentaje_of_nulls.to_csv("dumps_from_notebooks/null_porcentaje_" + table_name,index=False)



In [4]:
with open("../metadata/schema.json", "r") as f:
    schema = json.load(f)
print(schema)

{'application': {'SK_ID_CURR': {'type': 'categorical'}, 'TARGET': {'type': 'categorical'}, 'NAME_CONTRACT_TYPE': {'type': 'categorical'}, 'CODE_GENDER': {'type': 'categorical'}, 'FLAG_OWN_CAR': {'type': 'categorical'}, 'FLAG_OWN_REALTY': {'type': 'categorical'}, 'CNT_CHILDREN': {'type': 'numerical'}, 'AMT_INCOME_TOTAL': {'type': 'numerical'}, 'AMT_CREDIT': {'type': 'numerical'}, 'AMT_ANNUITY': {'type': 'numerical'}, 'AMT_GOODS_PRICE': {'type': 'numerical'}, 'NAME_TYPE_SUITE': {'type': 'categorical'}, 'NAME_INCOME_TYPE': {'type': 'categorical'}, 'NAME_EDUCATION_TYPE': {'type': 'categorical'}, 'NAME_FAMILY_STATUS': {'type': 'categorical'}, 'NAME_HOUSING_TYPE': {'type': 'categorical'}, 'REGION_POPULATION_RELATIVE': {'type': 'numerical'}, 'DAYS_BIRTH': {'type': 'numerical'}, 'DAYS_EMPLOYED': {'type': 'numerical'}, 'DAYS_REGISTRATION': {'type': 'numerical'}, 'DAYS_ID_PUBLISH': {'type': 'numerical'}, 'OWN_CAR_AGE': {'type': 'numerical'}, 'FLAG_MOBIL': {'type': 'categorical'}, 'FLAG_EMP_PHONE

In [19]:

def eda_per_table_printing_results(df: pd.DataFrame,schema: dict,table_name):
    results=eda_per_table(df,schema,table_name)
    for key,values in results.items() :
        print("--------------------------------------")
        print(key)
        print_dataframes(values)


def print_dataframes(dicts):
    for key,a_dataframe in dicts.items():
        print(key)
        display(a_dataframe)
    return


def eda_per_table(df: pd.DataFrame,schema: dict,table_name) -> dict :
    results={}
    for col in df.columns:
        dict_of_dataframes,column_name= eda_per_column(df,schema,table_name,col)
        results[column_name]=dict_of_dataframes
    return results


def eda_per_column(df: pd.DataFrame,schema: dict,table_name,column_name):
    dict_of_dataframes={}
    if(is_categorical(schema,table_name,column_name)):
        dict_of_dataframes= basic_eda_per_column_categorical(df,column_name)
    else:
        dict_of_dataframes=basic_eda_per_column_numerical(df,column_name) 
    return dict_of_dataframes,column_name




def is_categorical(schema: dict,table_name,column_name):
    type=schema[table_name][column_name]["type"]
    return type == "categorical"


def basic_eda_per_column_numerical(df: pd.DataFrame, column_name) -> dict: 
    column=df[column_name]
    dict_to_return={}
    basic_data_dict={}

    mean=column.mean()
    median=column.median()
    standar_deviation=column.std()

    basic_data_dict["min"]=column.min()
    basic_data_dict["max"]=column.max()
    basic_data_dict["mean"]=mean

    basic_data_dict["trim_mean"]= trim_mean(column.dropna(),proportiontocut=0.1)
    basic_data_dict["median"]= median
    basic_data_dict["standard_deviation"]=standar_deviation
    basic_data_dict["standard_error"]=column.sem()
    if(mean != 0):
        basic_data_dict["coefficient_of_variation"]= standar_deviation / abs(mean)

    basic_data_dataframe=pd.DataFrame([basic_data_dict])
    distribution_metrics_dataframe=pd.DataFrame([get_distribution_metrics(df,column_name)])

    if(column.isnull().sum() > 0):
        nulls_metrics_dataframe=pd.DataFrame([get_null_info(df,column_name)])
        dict_to_return["missings_metrics"]=nulls_metrics_dataframe

    dict_to_return["basic_data"]=basic_data_dataframe
    dict_to_return["distribution_metrics"]=distribution_metrics_dataframe

    return dict_to_return


def get_null_info(df: pd.DataFrame, column_name) -> dict:
    nulls_info={}
    
    column=df[column_name]
    column_null_maks=column.isnull()

    null_total=column_null_maks.sum()
    null_porcentaje= column_null_maks.mean() * 100

    rows_with_null=df[column_null_maks]
    target_correlation_nulls=rows_with_null["TARGET"].mean() * 100

    non_null_maks=~column.isnull()
    non_null_total=non_null_maks.sum()
    non_null_porcentaje=non_null_maks.mean() * 100

    non_null_values=df[non_null_maks]
    default_ratio_non_null= non_null_values["TARGET"].mean() * 100


    nulls_info["nulls_amount"]=null_total
    nulls_info["nulls_porcentaje"]=null_porcentaje
    nulls_info["default_ratio_nulls"]=target_correlation_nulls

    nulls_info["non_null_amount"] = non_null_total
    nulls_info["non_null_porcentaje"] = non_null_porcentaje
    nulls_info["non_null_default_ratio"] = default_ratio_non_null


    return nulls_info





def get_distribution_metrics(df: pd.DataFrame, column_name) -> dict:
    distribution_dict={}
    column=df[column_name]

    median= column.median()
    percentil_99=column.quantile(0.99)
    percentil_90=column.quantile(0.90)

    distribution_dict["skew"]=column.skew()
    distribution_dict["p90"]=percentil_90
    distribution_dict["p99"]=percentil_99

    if(median != 0):
        distribution_dict["ratio_p99_p50"]= percentil_99 / median
    if(percentil_90 !=0):
        distribution_dict["ratio_p99_p90"]= percentil_99 / percentil_90

    return distribution_dict
    

def basic_eda_per_column_categorical(df: pd.DataFrame,column_name) -> dict: 
    column=df[column_name]
    basic_data_dict={}
    cardinality=column.nunique(dropna=False)
    basic_data_dict["cardinality"]=cardinality
    basic_data_dict["mode"]=column.mode().to_list()
    dict_to_return={}
    dict_to_return["basic_data"]=pd.DataFrame([basic_data_dict])
    if(30 > cardinality):
        default_rate_per_category=(df.groupby(column_name,dropna=False)["TARGET"].mean() *100).reset_index(name="TARGET_RATE") 
        dict_to_return["default_rate"]=default_rate_per_category
    dict_to_return["frequency"]=get_counts_per_class(column)
    return dict_to_return


def get_counts_per_class(column : pd.Series):
    value_count_serie=column.value_counts(dropna=False)
    cardinality=value_count_serie.shape[0]
    if(cardinality < 1000):
        result_df= value_count_serie.reset_index()
        result_df.columns= ["CATEGORY", "COUNT"]
        result_df["SEGMENT"] = "full"
        return result_df
    head_df= value_count_serie.head(20).reset_index()
    head_df.columns= ["CATEGORY", "COUNT"]
    head_df["SEGMENT"] = "top"
    tail_df= value_count_serie.tail(20).reset_index()
    tail_df.columns= ["CATEGORY", "COUNT"]
    tail_df["SEGMENT"] = "bottom"
    resume_df=pd.concat([head_df,tail_df],ignore_index=True)
    return resume_df












    



In [8]:
eda_per_column(df,schema,"application","LOG_AMT_CREDIT")


KeyError: 'LOG_AMT_CREDIT'

In [ ]:
eda_per_table_printing_results(df,schema,"application")

In [ ]:
create_files_nulls_per_colmun(df,"aplication_train")

print(len(df))

In [ ]:

#now we will explore the hypothesis of train one model per type of contract. Starting for how viable is with the data available

print(df["NAME_CONTRACT_TYPE"].value_counts(normalize=True) * 100)
print(df.groupby("NAME_CONTRACT_TYPE")["TARGET"].mean() * 100)

#Considering the volume of the minoritary class (less than 10%) and their event ratio (5%) seems not ideal have to separate the models.



In [ ]:
#here i want to check if the main variables that describes the loan change their distribution and trending between the type of contract,
#and also see how much change in the case of default. 

df["LOG_AMT_CREDIT"]= np.log10(df["AMT_CREDIT"])

g = sns.displot(
    data=df,
    x="LOG_AMT_CREDIT",
    hue="NAME_CONTRACT_TYPE",
    bins=50,
    element="step",
    stat="density",
    col="TARGET",
    common_norm=False
)

for ax in g.axes.flat:
    ax.ticklabel_format(style='plain', axis='x')
    ax.ticklabel_format(style='plain', axis='y')

#default and the mont of the loan have negative correlation. In both type of contracts. That's a good proof about the similar behaivor between clases 
#and seems unnecesary split and train 2 diferent models.

In [ ]:
bins= np.arange(0,df["AMT_CREDIT"].max() + 100000, 100000)
df["BINED_AMT_CREDIT"]=  pd.qcut(df["AMT_CREDIT"],10)
df_groupBy_bins = df.groupby(["BINED_AMT_CREDIT","NAME_CONTRACT_TYPE"],observed=True)["TARGET"].agg(
    DEFAULT_RATE="mean",
    COUNT="size").reset_index()
df_groupBy_bins["CREDIT_BIN_CENTER"] = df_groupBy_bins["BINED_AMT_CREDIT"].apply(lambda x: x.mid)


gr=sns.relplot(data=df_groupBy_bins,x="CREDIT_BIN_CENTER",y="DEFAULT_RATE",kind="line",col="NAME_CONTRACT_TYPE")



In [ ]:
sns.scatterplot(
    data=df_groupBy_bins,
    x="CREDIT_BIN_CENTER",
    y="DEFAULT_RATE",
    hue="NAME_CONTRACT_TYPE",
    size="COUNT"
)


In [ ]:
"""About the idea of creating 2 different models for each type of contract: 

The amount of data and events is something to consider.
revolving loans represent 9.5% of observations and have a default ratio of 5%.
Therefore, train separate models significantly reduce the amount of total data and default events.

the analysis of the deciles of amount of credit and their respective default rate show similar overall trends for both contracts types, but have noticiable diferences in some segments.
For linear models this would require some features to model the interaction but in case of using tree based models the feature CONTRACT_TYPE 
seems enough

In conclusion, this preliminary analisis show there is no strong evidence supporting the need for two separate models. Even so, this hypothesis
will be tested formally later comparing model performance.
"""

In [ ]:
bureau_df=pd.read_csv("../data/bureau.csv")
bureau_df.head()


In [ ]:
create_files_nulls_per_colmun(bureau_df,"bureau")


In [ ]:
previous=pd.read_csv("../data/previous_application.csv")
previous.head()

In [ ]:
create_files_nulls_per_colmun(previous,"previous_aplication")
